 # Athena Validation
### Financial Distress Early Warning System

**Goal:** Validate predictions stored in S3 via Athena and create a deduplicated view for Power BI.

**Architecture:** S3 (predictions.csv) -> Athena External Table -> Clean View -> Power BI

In [1]:
%pip install awswrangler boto3

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3, time, pandas as pd

DATABASE = "bankruptcy_risk"
S3_OUTPUT = "s3://financial-distress-early-warning-eegunjobi/athena-results/"
athena = boto3.client("athena", region_name="us-east-1")

In [3]:
def run_ddl(sql):
    qid = athena.start_query_execution(
        QueryString=sql,
        ResultConfiguration={"OutputLocation": S3_OUTPUT}
    )["QueryExecutionId"]
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if status in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break
        time.sleep(1)
    print(status, ":", sql.splitlines()[0][:80])
    return status

def run_query(sql):
    qid = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE},
        ResultConfiguration={"OutputLocation": S3_OUTPUT}
    )["QueryExecutionId"]
    while True:
        resp = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        if resp["Status"]["State"] in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break
        time.sleep(1)
    if resp["Status"]["State"] != "SUCCEEDED":
        print("FAILED:", resp["Status"].get("StateChangeReason"))
        return None
    results = athena.get_query_results(QueryExecutionId=qid)
    rows = results["ResultSet"]["Rows"]
    header = [c["VarCharValue"] for c in rows[0]["Data"]]
    data = [[c.get("VarCharValue", "") for c in r["Data"]] for r in rows[1:]]
    return pd.DataFrame(data, columns=header)

## 1. Create External Table
Schema matches predictions.csv: company_id, risk_score, prediction, prediction_date, scored_at
Header row is skipped. Invalid data becomes NULL instead of failing the query.

In [5]:
# Drop if exists to ensure clean re-run
run_ddl("DROP TABLE IF EXISTS bankruptcy_risk.latest_predictions")

run_ddl("""
CREATE EXTERNAL TABLE bankruptcy_risk.latest_predictions (
  company_id STRING,
  risk_score DOUBLE,
  prediction STRING,
  prediction_date STRING,
  scored_at STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ('separatorChar' = ',', 'quoteChar' = '"', 'escapeChar' = '\\\\')
STORED AS TEXTFILE
LOCATION 's3://financial-distress-early-warning-eegunjobi/predictions/'
TBLPROPERTIES (
  'skip.header.line.count'='1',
  'use.null.for.invalid.data'='true'
)
""")

# Deduplication: Keep only latest scored_at per company_id
# This removes the 1 test row for company 3766 from 2026-11-12
run_ddl("""
CREATE OR REPLACE VIEW bankruptcy_risk.latest_predictions_clean AS
SELECT company_id, risk_score, prediction, prediction_date, scored_at
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY company_id ORDER BY scored_at DESC) AS rn
  FROM bankruptcy_risk.latest_predictions
)
WHERE rn = 1
""")

SUCCEEDED : DROP TABLE IF EXISTS bankruptcy_risk.latest_predictions
SUCCEEDED : 
SUCCEEDED : 


'SUCCEEDED'

In [6]:
print(run_query("SELECT COUNT(*) AS total_rows_raw FROM bankruptcy_risk.latest_predictions"))
print(run_query("SELECT COUNT(*) AS total_rows_clean FROM bankruptcy_risk.latest_predictions_clean"))
print(run_query("SELECT prediction, COUNT(*) AS cnt FROM bankruptcy_risk.latest_predictions_clean GROUP BY prediction"))
print(run_query("SELECT * FROM bankruptcy_risk.latest_predictions_clean WHERE company_id = '3766'"))
print(run_query("SELECT * FROM bankruptcy_risk.latest_predictions_clean ORDER BY risk_score DESC LIMIT 5"))

  total_rows_raw
0           1390
  total_rows_clean
0             1389
  prediction   cnt
0  High Risk   116
1   Low Risk  1273
  company_id risk_score prediction prediction_date                  scored_at
0       3766        0.0   Low Risk      2026-11-13  2026-11-13T07:23:12+00:00
  company_id risk_score prediction prediction_date                  scored_at
0       6699        1.0  High Risk      2026-11-13  2026-11-13T07:23:29+00:00
1       6718   0.999998  High Risk      2026-11-13  2026-11-13T07:23:12+00:00
2       6700   0.999995  High Risk      2026-11-13  2026-11-13T07:23:29+00:00
3       6711   0.999995  High Risk      2026-11-13  2026-11-13T07:23:29+00:00
4       6782   0.999992  High Risk      2026-11-13  2026-11-13T07:23:29+00:00
